# vLLM Helper Test Notebook

This notebook exercises the embed, generate, and reward pipelines using stubbed worker queues so you can observe both success and failure behaviour without launching the full vLLM stack.

The helper queues below simulate worker processes. If you have access to real models, you can replace the stubs with your actual `build_*` helpers to validate end-to-end runs.

In [ ]:
import json
import os
import queue
import tempfile
from typing import Dict, List, Optional, Sequence

import torch

from examples import vllm_embed, vllm_generate, vllm_reward2

def read_checkpoint_lines(path: str) -> List[Dict[str, object]]:
    if not path or not os.path.exists(path):
        return []
    with open(path, 'r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def make_fake_embed_model(
    outputs_by_batch: Dict[int, Sequence[Sequence[float]]],
    error_batches: Optional[Sequence[int]] = None,
) -> vllm_embed.EmbeddingWorkerModel:
    result_q: 'queue.Queue[object]' = queue.Queue()

    class FakeQueue:
        def __init__(self, result_q, outputs, errors):
            self._result_q = result_q
            self._outputs = outputs
            self._errors = set(errors or [])
        def put(self, message):
            # init messages ignored in stub
            return None
        def put_nowait(self, message):
            job_id, batch_idx, kind, payload = message
            self._result_q.put(('__log__', 0, {'msg': f'fake worker accepted batch {batch_idx}'}))
            if batch_idx in self._errors:
                self._result_q.put((job_id, batch_idx, {'error': 'RuntimeError: forced failure', 'wid': 0}))
            else:
                embeds = self._outputs[batch_idx]
                self._result_q.put((job_id, batch_idx, {'embeddings': embeds, 'wid': 0}))
    fake_queue = FakeQueue(result_q, outputs_by_batch, error_batches)
    model = vllm_embed.EmbeddingWorkerModel.__new__(vllm_embed.EmbeddingWorkerModel)
    model.task_queues = [fake_queue]
    model.result_q = result_q
    model._rr = 0
    model.ctx = None
    model.model = 'fake-model'
    model.device_groups = [[0]]
    model.llm_kwargs = {}
    model.output_to_cpu = True
    model.procs = []
    return model

def make_fake_generate_pool(
    outputs_by_batch: Dict[int, Sequence[str]],
    error_batches: Optional[Sequence[int]] = None,
) -> vllm_generate.LLMWorkerPool:
    result_q: 'queue.Queue[object]' = queue.Queue()

    class FakeQueue:
        def __init__(self, result_q, outputs, errors):
            self._result_q = result_q
            self._outputs = outputs
            self._errors = set(errors or [])
        def put(self, message):
            job_id, batch_idx, kind, payload = message
            if batch_idx in self._errors:
                self._result_q.put((job_id, batch_idx, {'error': 'RuntimeError: forced failure'}))
            else:
                texts = list(self._outputs[batch_idx])
                self._result_q.put((job_id, batch_idx, {'texts': texts}))
    fake_queue = FakeQueue(result_q, outputs_by_batch, error_batches)
    pool = vllm_generate.LLMWorkerPool.__new__(vllm_generate.LLMWorkerPool)
    pool.task_queues = [fake_queue]
    pool.result_q = result_q
    pool._rr = 0
    pool.ctx = None
    pool.model = 'fake-model'
    pool.device_groups = [[0]]
    pool.llm_kwargs = {}
    pool.procs = []
    return pool

def make_fake_reward_worker(
    outputs_by_batch: Dict[int, Sequence[torch.Tensor]],
    error_batches: Optional[Sequence[int]] = None,
) -> vllm_reward2.LLMWorker:
    result_q: 'queue.Queue[object]' = queue.Queue()

    class FakeQueue:
        def __init__(self, result_q, outputs, errors):
            self._result_q = result_q
            self._outputs = outputs
            self._errors = set(errors or [])
        def put(self, message):
            # init messages ignored in stub
            return None
        def put_nowait(self, message):
            job_id, batch_idx, kind, payload = message
            self._result_q.put(('__log__', -1, 0, {'msg': f'fake worker accepted batch {batch_idx}'}))
            if batch_idx in self._errors:
                self._result_q.put((job_id, batch_idx, 0, {'error': 'RuntimeError: forced failure'}))
            else:
                outputs = self._outputs[batch_idx]
                self._result_q.put((job_id, batch_idx, 0, {'outputs': outputs}))
    fake_queue = FakeQueue(result_q, outputs_by_batch, error_batches)
    worker = vllm_reward2.LLMWorker.__new__(vllm_reward2.LLMWorker)
    worker.task_queues = [fake_queue]
    worker.result_q = result_q
    worker._rr = 0
    worker.ctx = None
    worker.model = 'fake-model'
    worker.device_groups = [[0]]
    worker.llm_kwargs = {}
    worker.requests = []
    worker.tokenizer = None
    worker.procs = []
    return worker


## Embedding helper (`examples.vllm_embed`)

In [ ]:
inputs = ['alpha prompt', 'beta prompt']
embed_outputs = {
    0: [[0.1, 0.2]],
    1: [[0.3, 0.4]],
}
embed_model = make_fake_embed_model(embed_outputs)
with tempfile.TemporaryDirectory() as tmpdir:
    checkpoint_path = os.path.join(tmpdir, 'embed_success.jsonl')
    vectors = embed_model.embed(inputs, batch_size=1, checkpoint_path=checkpoint_path)
    print('Returned embeddings:', vectors)
    print('Checkpoint entries:', read_checkpoint_lines(checkpoint_path))


In [ ]:
embed_error_model = make_fake_embed_model({0: [[0.5, 0.6]]}, error_batches=[0])
try:
    embed_error_model.embed(['error prompt'], batch_size=1, checkpoint_path=None)
except RuntimeError as exc:
    print('Caught expected exception:', exc)


## Generation helper (`examples.vllm_generate`)

In [ ]:
prompts = ['first prompt', 'second prompt', 'third prompt']
generate_outputs = {
    0: ['output-1', 'output-2'],
    1: ['output-3'],
}
pool = make_fake_generate_pool(generate_outputs)
with tempfile.TemporaryDirectory() as tmpdir:
    checkpoint_path = os.path.join(tmpdir, 'generate_success.jsonl')
    texts = pool.generate(prompts, batch_size=2, checkpoint_path=checkpoint_path)
    print('Returned texts:', texts)
    print('Checkpoint entries:', read_checkpoint_lines(checkpoint_path))


In [ ]:
pool_error = make_fake_generate_pool({0: ['ok response']}, error_batches=[0])
try:
    pool_error.generate(['bad prompt'], batch_size=1)
except RuntimeError as exc:
    print('Caught expected exception:', exc)


## Reward helper (`examples.vllm_reward2`)

In [ ]:
reward_prompts = ['query-1', 'query-2', 'query-3']
reward_outputs = {
    0: torch.tensor([0.8, -0.2]),
    1: torch.tensor([0.5]),
}
reward_worker = make_fake_reward_worker(reward_outputs)
with tempfile.TemporaryDirectory() as tmpdir:
    checkpoint_path = os.path.join(tmpdir, 'reward_success.jsonl')
    scores = reward_worker.encode(reward_prompts, batch_size=2, checkpoint_path=checkpoint_path)
    print('Returned scores:', scores.tolist())
    print('Checkpoint entries:', read_checkpoint_lines(checkpoint_path))


In [ ]:
reward_error = make_fake_reward_worker({0: torch.tensor([0.1])}, error_batches=[0])
try:
    reward_error.encode(['should fail'], batch_size=1)
except RuntimeError as exc:
    print('Caught expected exception:', exc)
